## Reproducing result figures and evaluation metrics

This notebook can be used to evaluate the different models by running new simulations and computing metrics and associated figures:
* A priori metrics: $pdf$ of $\tau_\omega$ and $\partial_t \partial E(k)_\text{sgs}$
* Evolution of integral quantities: $E(t)$ and $Z(t)$
* Spectral metrics: $E(k)$ and enstrophy $Z(k)$
* Distributional metrics: $pdf(\omega)$ and $pdf(E)$

The notebook also computes similarity scores for each of these metrics.

In [ ]:
import sys, os
sys.path.append(os.path.dirname(os.getcwd()))
import warnings

import tqdm
import h5py
import matplotlib.pyplot as plt

plt.rcParams.update({
  'mathtext.fontset': 'cm'
})

import numpy as np
import jax
import jax.numpy as jnp
import jax.random as jnr
import scipy

jax.config.update(
  'jax_enable_x64', True
)

from flax import nnx
import orbax.checkpoint as ocp

import models.time_solver as stepper
from models.qg_periodic import (
    QgPeriodic, 
    divergence,
    kinetic_energy,
    enstrophy,
    iso_spectrum,
)
from models.cnn import (
    FwdCNN,
)
from utils import (
    into_s,
    from_s,
    spectral_pad,
)

In [ ]:
cfg_name = 'qg-det'
data_path = os.path.join(os.path.join(os.getcwd(), '../data'), cfg_name)

with h5py.File(os.path.join(data_path, 'datasets.h5'), 'r') as f:
    ratio = f.attrs['ratio']
eq, t0, om_s = QgPeriodic.load(
    os.path.join(data_path, 'snapshot.h5')
)

eq_coarse = QgPeriodic(
    nu=eq.nu,
    mu=eq.mu,
    beta=eq.beta,
    n_kx=int((eq.n_kx - 1) / ratio + 1),
    n_ky=int(eq.n_ky / ratio)
)
print(eq_coarse)

with h5py.File(os.path.join(data_path, 'datasets.h5'), 'r') as f:
    dt = f.attrs['dt']
    n_steps = f.attrs['n_steps']
    n_trajs = f.attrs['n_trajs']

    sigma = f.attrs['sigma']
    k_f = f.attrs['k_f']

    om_c_emu = np.array(f['om_c_emu'])

iso_K = np.sqrt(eq_coarse.lap)

forcing_spectrum = np.full_like(iso_K, sigma)
forcing_spectrum[iso_K < k_f - 1] = 0
forcing_spectrum[iso_K > k_f + 1] = 0
forcing_spectrum[iso_K == 0] = 0

forcing_det = into_s(np.cos(k_f * eq_coarse.X) + np.cos(k_f * eq_coarse.Y))

### Computing the entire set of metrics and similarity scores:

In [ ]:
abstract_model = nnx.eval_shape(lambda: FwdCNN(
    in_features=1,
    latent=64,
    kernel_size=5,
    out_features=1,
    n_blocks=5,
    means=jnp.zeros_like(eq_coarse.lap), 
    stds=jnp.zeros_like(eq_coarse.lap),
    activation=nnx.relu,
    rngs=nnx.Rngs(42)
))

with h5py.File(os.path.join(data_path, 'datasets.h5'), 'r') as f:
    tau_x_ref = np.array(f['tau_x_ref'])
    tau_y_ref = np.array(f['tau_y_ref'])
tau_g = from_s(divergence(
    eq_coarse, 
    tau_x_ref[-1, -1], 
    tau_y_ref[-1, -1]
))

tau_range = 1.5 * max(tau_g.max(), abs(tau_g.min()))
tau_num = 500
tau_sample = np.linspace(-tau_range, tau_range, num=tau_num)
    
graph, abstract_state = nnx.split(abstract_model)
def nnx_model(checkpoint_path):
    checkpointer = ocp.Checkpointer(ocp.StandardCheckpointHandler())
    with warnings.catch_warnings(action='ignore'):
        state = checkpointer.restore(checkpoint_path, abstract_state)
    return nnx.merge(
        graph, 
        state
    )

def apriori_metrics(
    name: str,
    dns_path: str,
    file_path: str,
    model: nnx.Module,
    eq: QgPeriodic
):
    save_path = file_path + '.apriori_metrics.npz'
    if os.path.isfile(save_path):
        return np.load(save_path)
    else:
        metrics = {
            'mape_s': [],
            'r2_coef_s': [],
            'corr_s': [],
            'pdf_s': [],
            'flux_s': [],
        }

        with h5py.File(dns_path, 'r') as f:
            time = np.array(f['time'])
            samples = len(time)
            samples_digits = f.attrs['digits']
            print('Computing apriori metrics for {} ({} samples)'.format(name, str(samples)))
            pbar = tqdm.tqdm(range(samples), bar_format='{l_bar}{bar:10}{r_bar}{bar:-10b}')
            for i in pbar:
                i_str = str(i).zfill(samples_digits)

                om_s = spectral_pad(np.array(f['om_s_' + i_str]), eq.n_kx, eq.n_ky)
                ps_s = -eq.ilap * om_s
                
                tau_s = np.array(f['tau_s_' + i_str])
                tau_g = from_s(tau_s)

                pred_g = model(jnp.expand_dims(from_s(om_s), (0,-1))).squeeze()
                pred_s = into_s(pred_g)

                epsilon = np.asarray(np.finfo(np.float64).eps)
                metrics['mape_s'].append(
                    np.mean(np.abs(pred_g - tau_g) / np.maximum(np.abs(tau_g), epsilon))
                )

                num = np.sum((tau_g - pred_g)**2)
                den = np.sum((tau_g - np.mean(tau_g))**2)
                metrics['r2_coef_s'].append(
                    1 - num / den
                )

                corr, p_value = scipy.stats.pearsonr(tau_g.flatten(), pred_g.flatten())
                metrics['corr_s'].append(
                    corr
                )

                hist, _ = np.histogram(pred_g, bins=tau_sample, density=True)
                metrics['pdf_s'].append(
                    hist
                )
                
                _, _, flux = iso_spectrum(eq, -np.real(np.conj(ps_s) * pred_s))
                metrics['flux_s'].append(
                    flux
                )
        
        np.savez(
            save_path, **metrics
        )
        return (
            metrics
        )

def metrics(
    name: str,
    file_path: str,
    eq: QgPeriodic,
    process = None,
    compute_sgs_f = False,
):
    if process:
        process_name, process_fn = process
    save_path = file_path + ('.' + process_name + '.metrics.npz' if process else '.metrics.npz')
    if os.path.isfile(save_path):
        return np.load(save_path)
    else:
        metrics = {
            'eke_t':  [],
            'ens_t':  [],
            'ek_k_t': [],
            'en_k_t': [],
            'sgs_flux_t': [],
            'sgs_pdf_t': []
        }

        with h5py.File(file_path, 'r') as f:
            time = np.array(f['time'])
            samples = len(time)
            samples_digits = f.attrs['digits']
            print('Computing metrics for {} ({} samples)'.format(name, str(samples)))
            pbar = tqdm.tqdm(range(samples), bar_format='{l_bar}{bar:10}{r_bar}{bar:-10b}')
            for i in pbar:
                i_str = str(i).zfill(samples_digits)

                om_s = np.array(f['om_s_' + i_str])
                if process:
                    om_s = process_fn(om_s)

                ps_s = -eq.ilap * om_s
                ux_s = -1j * eq.k_y * ps_s
                uy_s =  1j * eq.k_x * ps_s

                metrics['eke_t'].append(
                    kinetic_energy(ux_s, uy_s)
                )
                metrics['ens_t'].append(
                    enstrophy(om_s)
                )

                kr, bin_max, ek_k = iso_spectrum(eq, eq.lap * np.abs(ps_s)**2)
                metrics['ek_k_t'].append(
                    ek_k
                )

                metrics['en_k_t'].append(
                    kr**2 * ek_k
                )

                if compute_sgs_f:
                    tau_s = np.array(f['tau_s_' + i_str])
                    hist, edges = np.histogram(from_s(tau_s), bins=tau_sample, density=True)
                    metrics['edges'] = edges[:-1] + tau_range / tau_num
                    metrics['sgs_pdf_t'].append(
                        hist
                    )
                    
                    _, _, flux = iso_spectrum(eq, -np.real(np.conj(ps_s) * tau_s))
                    metrics['sgs_flux_t'].append(
                        flux
                    )

        metrics['kr'] = kr
        metrics['bin_max'] = bin_max
        metrics['time'] = time - t0
        metrics['om_s'] = om_s
        metrics['ps_s'] = ps_s
        metrics['ux_s'] = ux_s
        metrics['uy_s'] = uy_s
        
        np.savez(
            save_path, **metrics
        )
        return (
            metrics
        )

def similarity(
    ref_m,
    nop_m,
    cur_m,
    eq: QgPeriodic,
):
    def __diff_rmse__(x, y):
        return np.sqrt(np.mean(np.square(np.array(x) - np.array(y))))
    def __diff_dist__(u, v):
        return scipy.stats.wasserstein_distance(u, v)
    return {
        'eke': 1 - __diff_rmse__(cur_m['eke_t'], ref_m['eke_t']) / __diff_rmse__(nop_m['eke_t'], ref_m['eke_t']),
        'ens': 1 - __diff_rmse__(cur_m['ens_t'], ref_m['ens_t']) / __diff_rmse__(nop_m['ens_t'], ref_m['ens_t']),
        'ek_k': 1 - __diff_rmse__(cur_m['ek_k_t'], ref_m['ek_k_t']) / __diff_rmse__(nop_m['ek_k_t'], ref_m['ek_k_t']),
        'en_k': 1 - __diff_rmse__(cur_m['en_k_t'], ref_m['en_k_t']) / __diff_rmse__(nop_m['en_k_t'], ref_m['en_k_t']),
        'pdf_om': 1 - __diff_dist__(
            from_s(cur_m['om_s']).flatten(), 
            from_s(ref_m['om_s']).flatten()
        ) / __diff_dist__(
            from_s(nop_m['om_s']).flatten(), 
            from_s(ref_m['om_s']).flatten()
        ),
        'pdf_ek': 1 - __diff_dist__(
            0.5 * ((from_s(cur_m['ux_s']) + from_s(cur_m['uy_s']))**2).flatten(), 
            0.5 * ((from_s(ref_m['ux_s']) + from_s(ref_m['uy_s']))**2).flatten()
        ) / __diff_dist__(
            0.5 * ((from_s(nop_m['ux_s']) + from_s(nop_m['uy_s']))**2).flatten(), 
            0.5 * ((from_s(ref_m['ux_s']) + from_s(ref_m['uy_s']))**2).flatten()
        )
    }

storage = '/hoyt/PROJECTS/pr-data-ocean/frezath/gradient-free-subgrid-neural-emulator/'
storage_path = os.path.join(storage, cfg_name)

metrics_dns = metrics(
    name='DNS',
    file_path=os.path.join(storage_path, 'logs_dns.h5'),
    eq=eq,
)

metrics_fdns = metrics(
    name='fDNS',
    file_path=os.path.join(storage_path, 'logs_dns.h5'),
    eq=eq_coarse,
    process=('coarse', lambda om_s: spectral_pad(om_s, eq_coarse.n_kx, eq_coarse.n_ky)),
    compute_sgs_f=True
)

metrics_nop = metrics(
    name='No model',
    file_path=os.path.join(storage_path, 'logs_nop.h5'),
    eq=eq_coarse,
)

apriori_metrics_off = apriori_metrics(
    name='Offline',
    file_path=os.path.join(storage_path, 'logs_off.h5'),
    dns_path=os.path.join(storage_path, 'logs_dns.h5'),
    model=nnx_model(os.path.join(data_path, 'off_checkpoint/')),
    eq=eq_coarse
)

metrics_off = metrics(
    name='Offline',
    file_path=os.path.join(storage_path, 'logs_off.h5'),
    eq=eq_coarse
)

#similarity_off = similarity(
#    metrics_fdns,
#    metrics_nop,
#    metrics_off,
#    eq
#)

apriori_metrics_state_ref = apriori_metrics(
    name='Reference (state, online)',
    file_path=os.path.join(storage_path, 'logs_state_ref_ek.h5'),
    dns_path=os.path.join(storage_path, 'logs_dns.h5'),
    model=nnx_model(os.path.join(data_path, 'state_ref_ek_checkpoint/')),
    eq=eq_coarse
)

metrics_state_ref = metrics(
    name='Reference (state, online)',
    file_path=os.path.join(storage_path, 'logs_state_ref_ek.h5'),
    eq=eq_coarse
)

similarity_state_ref = similarity(
    metrics_fdns,
    metrics_nop,
    metrics_state_ref,
    eq
)

apriori_metrics_subgrid_ref = apriori_metrics(
    name='Reference (subgrid, online)',
    file_path=os.path.join(storage_path, 'logs_subgrid_ref_mse.h5'),
    dns_path=os.path.join(storage_path, 'logs_dns.h5'),
    model=nnx_model(os.path.join(data_path, 'subgrid_ref_mse_checkpoint/')),
    eq=eq_coarse
)

metrics_subgrid_ref = metrics(
    name='Reference (subgrid, online)',
    file_path=os.path.join(storage_path, 'logs_subgrid_ref_mse.h5'),
    eq=eq_coarse
)

similarity_subgrid_ref = similarity(
    metrics_fdns,
    metrics_nop,
    metrics_subgrid_ref,
    eq
)

apriori_metrics_state_small = apriori_metrics(
    name='State (small)',
    file_path=os.path.join(storage_path, 'logs_state_small_ek.h5'),
    dns_path=os.path.join(storage_path, 'logs_dns.h5'),
    model=nnx_model(os.path.join(data_path, 'state_small_ek_checkpoint/')),
    eq=eq_coarse
)

metrics_state_small = metrics(
    name='State (small)',
    file_path=os.path.join(storage_path, 'logs_state_small_ek.h5'),
    eq=eq_coarse
)

#similarity_state_small = similarity(
#    metrics_fdns,
#    metrics_nop,
#    metrics_state_small,
#    eq
#)

apriori_metrics_subgrid_small = apriori_metrics(
    name='Subgrid (small)',
    file_path=os.path.join(storage_path, 'logs_subgrid_small_mse.h5'),
    dns_path=os.path.join(storage_path, 'logs_dns.h5'),
    model=nnx_model(os.path.join(data_path, 'subgrid_small_mse_checkpoint/')),
    eq=eq_coarse
)

metrics_subgrid_small = metrics(
    name='Subgrid (small)',
    file_path=os.path.join(storage_path, 'logs_subgrid_small_mse.h5'),
    eq=eq_coarse
)

similarity_subgrid_small = similarity(
    metrics_fdns,
    metrics_nop,
    metrics_subgrid_small,
    eq
)

apriori_metrics_state_large = apriori_metrics(
    name='State (large)',
    file_path=os.path.join(storage_path, 'logs_state_large_ek.h5'),
    dns_path=os.path.join(storage_path, 'logs_dns.h5'),
    model=nnx_model(os.path.join(data_path, 'state_large_ek_checkpoint/')),
    eq=eq_coarse
)

metrics_state_large = metrics(
    name='State (large)',
    file_path=os.path.join(storage_path, 'logs_state_large_ek.h5'),
    eq=eq_coarse
)

similarity_state_large = similarity(
    metrics_fdns,
    metrics_nop,
    metrics_state_large,
    eq
)

apriori_metrics_subgrid_large = apriori_metrics(
    name='Subgrid (large)',
    file_path=os.path.join(storage_path, 'logs_subgrid_large_mse.h5'),
    dns_path=os.path.join(storage_path, 'logs_dns.h5'),
    model=nnx_model(os.path.join(data_path, 'subgrid_large_mse_checkpoint/')),
    eq=eq_coarse
)

metrics_subgrid_large = metrics(
    name='Subgrid (large)',
    file_path=os.path.join(storage_path, 'logs_subgrid_large_mse.h5'),
    eq=eq_coarse
)

similarity_subgrid_large = similarity(
    metrics_fdns,
    metrics_nop,
    metrics_subgrid_large,
    eq
)

### A priori metrics $pdf(\tau_\omega)$ and $\partial_t \partial E(k)_\text{sgs}$:

In [ ]:
fig, axs = plt.subplots(ncols=2, nrows=1, figsize=(7.5, 3.5), dpi=120)

edges = metrics_fdns['edges']

axs[0].semilogy(edges, np.mean(metrics_fdns['sgs_pdf_t'], axis=0), color='k', label=r'$\overline{\text{DNS}}$')
axs[0].semilogy(edges, np.mean(apriori_metrics_off['pdf_s'], axis=0), label=r'$\tau \equiv \mathcal{M}_\text{off}$', color='#009E73')
axs[0].semilogy(edges, np.mean(apriori_metrics_state_small['pdf_s'], axis=0), label=r'$\tau \equiv \mathcal{M}_\text{on}^\text{state} (\text{sm})$', color='#E69F00', linestyle='--')
axs[0].semilogy(edges, np.mean(apriori_metrics_subgrid_small['pdf_s'], axis=0), label=r'$\tau \equiv \mathcal{M}_\text{on}^\text{subgrid} (\text{sm})$', color='#E69F00')
axs[0].semilogy(edges, np.mean(apriori_metrics_state_large['pdf_s'], axis=0), label=r'$\tau \equiv \mathcal{M}_\text{on}^\text{state} (\text{lg})$', color='#D55E00', linestyle='--')
axs[0].semilogy(edges, np.mean(apriori_metrics_subgrid_large['pdf_s'], axis=0), label=r'$\tau \equiv \mathcal{M}_\text{on}^\text{subgrid} (\text{lg})$', color='#D55E00')
axs[0].semilogy(edges, np.mean(apriori_metrics_state_ref['pdf_s'], axis=0), label=r'$\tau \equiv \mathcal{M}_\text{on}^\text{state} (\text{ref})$', color='#56B4E9', linestyle='--')
axs[0].semilogy(edges, np.mean(apriori_metrics_subgrid_ref['pdf_s'], axis=0), label=r'$\tau \equiv \mathcal{M}_\text{on}^\text{subgrid} (\text{ref})$', color='#56B4E9')

x1, x2, y1, y2 = -15, 15, 1e-2, 1.5e-2
axins = axs[0].inset_axes([0.05, 0.6, 0.35, 0.35], xlim=(x1, x2), ylim=(y1, y2))
axins.semilogy(edges, np.mean(metrics_fdns['sgs_pdf_t'], axis=0), color='k', label=r'$\overline{\text{DNS}}$')
axins.semilogy(edges, np.mean(apriori_metrics_off['pdf_s'], axis=0), label=r'$\tau \equiv \mathcal{M}_\text{off}$', color='#009E73')
axins.semilogy(edges, np.mean(apriori_metrics_subgrid_small['pdf_s'], axis=0), label=r'$\tau \equiv \mathcal{M}_\text{on}^\text{subgrid} (\text{sm})$', color='#E69F00')
axins.semilogy(edges, np.mean(apriori_metrics_subgrid_large['pdf_s'], axis=0), label=r'$\tau \equiv \mathcal{M}_\text{on}^\text{subgrid} (\text{lg})$', color='#D55E00')
axins.semilogy(edges, np.mean(apriori_metrics_subgrid_ref['pdf_s'], axis=0), label=r'$\tau \equiv \mathcal{M}_\text{on}^\text{subgrid} (\text{ref})$', color='#56B4E9')
axins.tick_params(axis='both', which='both', labelleft=False, labelbottom=False)
axs[0].indicate_inset_zoom(axins, edgecolor='black')

axs[0].set_ylim(bottom=1e-6)
axs[0].set_xlabel(r'$\tau_\omega$', fontsize=15)
axs[0].set_ylabel(r'$pdf$', fontsize=15)
#axs[0].legend(fontsize=10, frameon=False)
axs[0].tick_params(reset=True, axis='both', which='both', direction='in')

kr_c = metrics_nop['kr']

axs[1].semilogx(kr_c, np.mean(metrics_fdns['sgs_flux_t'], axis=0), color='k', label=r'$\overline{\text{DNS}}$')
axs[1].axhline(0, label=r'$\tau \equiv 0$', linestyle='--', color='#999999')
axs[1].semilogx(kr_c, np.mean(apriori_metrics_off['flux_s'], axis=0), label=r'$\tau \equiv \mathcal{M}_{\text{off}}$', color='#009E73')
axs[1].semilogx(kr_c, np.mean(apriori_metrics_state_small['flux_s'], axis=0), label=r'$\tau \equiv \mathcal{M}_\text{on}^\text{state} (\text{sm})$', color='#E69F00', linestyle='--')
axs[1].semilogx(kr_c, np.mean(apriori_metrics_subgrid_small['flux_s'], axis=0), label=r'$\tau \equiv \mathcal{M}_\text{on}^\text{subgrid} (\text{sm})$', color='#E69F00')
axs[1].semilogx(kr_c, np.mean(apriori_metrics_state_large['flux_s'], axis=0), label=r'$\tau \equiv \mathcal{M}_\text{on}^\text{state} (\text{lg})$', color='#D55E00', linestyle='--')
axs[1].semilogx(kr_c, np.mean(apriori_metrics_subgrid_large['flux_s'], axis=0), label=r'$\tau \equiv \mathcal{M}_\text{on}^\text{subgrid} (\text{lg})$', color='#D55E00')
axs[1].semilogx(kr_c, np.mean(apriori_metrics_state_ref['flux_s'], axis=0), label=r'$\tau \equiv \mathcal{M}_{\text{on}}^\text{state} (\text{ref})$', color='#56B4E9', linestyle='--')
axs[1].semilogx(kr_c, np.mean(apriori_metrics_subgrid_ref['flux_s'], axis=0), label=r'$\tau \equiv \mathcal{M}_{\text{on}}^\text{subgrid} (\text{ref})$', color='#56B4E9')

x1, x2, y1, y2 = 0.6, 110, -0.00025, 0.0005
axins = axs[1].inset_axes([0.05, 0.3, 0.35, 0.35], xlim=(x1, x2), ylim=(y1, y2))
axins.semilogx(kr_c, np.mean(metrics_fdns['sgs_flux_t'], axis=0), color='k', label=r'$\overline{\text{DNS}}$')
axins.axhline(0, label=r'$\tau \equiv 0$', linestyle='--', color='#999999')
axins.semilogx(kr_c, np.mean(apriori_metrics_state_ref['flux_s'], axis=0), label=r'$\tau \equiv \mathcal{M}_{\text{on}}^\text{state} (\text{ref})$', color='#56B4E9', linestyle='--')
axins.tick_params(axis='both', which='both', labelleft=False, labelbottom=False)
axs[1].indicate_inset_zoom(axins, edgecolor='black')

axs[1].set_xlabel(r'$k$', fontsize=15)
axs[1].set_ylabel(r'$\partial_t \partial E(k)_\text{sgs}$', fontsize=15)
axs[1].legend(fontsize=10, frameon=False)
axs[1].tick_params(reset=True, axis='both', which='both', direction='in')

fig.tight_layout()
fig.savefig(os.path.join(data_path, 'apriori_eval.pdf'))
plt.show()

### A posteriori temporal evolution of kinetic energy $E(t)$ and enstrophy $Z(t)$:

In [ ]:
fig, axs = plt.subplots(ncols=2, nrows=1, figsize=(7.0, 3.5), dpi=120)

axs[0].plot(metrics_fdns['time'], metrics_fdns['eke_t'], color='k', label=r'$\overline{\text{DNS}}$')
axs[0].plot(metrics_nop['time'], metrics_nop['eke_t'], label=r'$\tau \equiv 0$', linestyle='--', color='#999999')
axs[0].plot(metrics_off['time'], metrics_off['eke_t'], label=r'$\tau \equiv \mathcal{M}_{\text{off}}$', color='#009E73')
axs[0].plot(metrics_state_small['time'], metrics_state_small['eke_t'], label=r'$\tau \equiv \mathcal{M}_\text{on}^\text{state} (\text{sm})$', color='#E69F00', linestyle='--')
axs[0].plot(metrics_state_large['time'], metrics_state_large['eke_t'], label=r'$\tau \equiv \mathcal{M}_\text{on}^\text{state} (\text{lg})$', color='#D55E00', linestyle='--')
axs[0].plot(metrics_state_ref['time'], metrics_state_ref['eke_t'], label=r'$\tau \equiv \mathcal{M}_{\text{on}}^\text{state} (\text{ref})$', color='#56B4E9', linestyle='--')
axs[0].plot(metrics_subgrid_small['time'], metrics_subgrid_small['eke_t'], label=r'$\tau \equiv \mathcal{M}_\text{on}^\text{subgrid} (\text{sm})$', color='#E69F00')
axs[0].plot(metrics_subgrid_large['time'], metrics_subgrid_large['eke_t'], label=r'$\tau \equiv \mathcal{M}_\text{on}^\text{subgrid} (\text{lg})$', color='#D55E00')
axs[0].plot(metrics_subgrid_ref['time'], metrics_subgrid_ref['eke_t'], label=r'$\tau \equiv \mathcal{M}_{\text{on}}^\text{subgrid} (\text{ref})$', color='#56B4E9')

axs[0].set_xlabel(r'$t$', fontsize=15)
axs[0].set_ylabel(r'$E(t)$', fontsize=15)
axs[0].tick_params(reset=True, axis='both', which='both', direction='in')
leg = fig.legend(fontsize=10, frameon=False, ncols=3, loc='upper center', bbox_to_anchor=(0.4,1.2))

axs[1].plot(metrics_fdns['time'], metrics_fdns['ens_t'], color='k', label=r'$\overline{\text{DNS}}$')
axs[1].plot(metrics_nop['time'], metrics_nop['ens_t'], label=r'$\tau \equiv 0$', linestyle='--', color='#999999')
axs[1].plot(metrics_off['time'], metrics_off['ens_t'], label=r'$\tau \equiv \mathcal{M}_{\text{off}}$', color='#009E73')
axs[1].plot(metrics_state_small['time'], metrics_state_small['ens_t'], label=r'$\tau \equiv \mathcal{M}_\text{on}^\text{state} (\text{sm})$', color='#E69F00', linestyle='--')
axs[1].plot(metrics_state_large['time'], metrics_state_large['ens_t'], label=r'$\tau \equiv \mathcal{M}_\text{on}^\text{state} (\text{lg})$', color='#D55E00', linestyle='--')
axs[1].plot(metrics_state_ref['time'], metrics_state_ref['ens_t'], label=r'$\tau \equiv \mathcal{M}_{\text{on}}^\text{state} (\text{ref})$', color='#56B4E9', linestyle='--')
axs[1].plot(metrics_subgrid_small['time'], metrics_subgrid_small['ens_t'], label=r'$\tau \equiv \mathcal{M}_\text{on}^\text{subgrid} (\text{sm})$', color='#E69F00')
axs[1].plot(metrics_subgrid_large['time'], metrics_subgrid_large['ens_t'], label=r'$\tau \equiv \mathcal{M}_\text{on}^\text{subgrid} (\text{lg})$', color='#D55E00')
axs[1].plot(metrics_subgrid_ref['time'], metrics_subgrid_ref['ens_t'], label=r'$\tau \equiv \mathcal{M}_{\text{on}}^\text{subgrid} (\text{ref})$', color='#56B4E9')

axs[1].set_xlabel(r'$t$', fontsize=15)
axs[1].set_ylabel(r'$Z(t)$', fontsize=15)
axs[1].set_ylim(bottom=5)
#axs[1].legend(fontsize=10, frameon=False)
axs[1].tick_params(reset=True, axis='both', which='both', direction='in')

fig.tight_layout()
fig.savefig(os.path.join(data_path, 'integrals_evolution.pdf'), bbox_inches='tight')
plt.show()

### Vorticity fields at the end of the a posteriori evaluation:

In [ ]:
fig, axs = plt.subplots(ncols=3, nrows=3, figsize=(9.0, 8.5))

axs[0,0].contourf(eq.X, eq.Y, from_s(metrics_dns['om_s']), cmap='twilight', levels=45, vmin=-40, vmax=40)
axs[0,0].set_title(r'$\text{DNS}$', fontsize=15)
contour = axs[0,1].contourf(eq_coarse.X, eq_coarse.Y, from_s(metrics_fdns['om_s']), cmap='twilight', levels=45, vmin=-40, vmax=40)
axs[0,1].set_title(r'$\overline{\text{DNS}}$', fontsize=15)
axs[0,2].contourf(eq_coarse.X, eq_coarse.Y, from_s(metrics_nop['om_s']), cmap='twilight', levels=45, vmin=-40, vmax=40)
axs[0,2].set_title(r'$\tau \equiv 0$', fontsize=15)

axs[1,0].contourf(eq_coarse.X, eq_coarse.Y, from_s(metrics_state_small['om_s']) * np.nan, cmap='twilight', levels=45, vmin=-40, vmax=40)
axs[1,0].set_title(r'$\tau \equiv \mathcal{M}_\text{on}^\text{state} (\text{sm})$', fontsize=15)
axs[1,1].contourf(eq_coarse.X, eq_coarse.Y, from_s(metrics_state_large['om_s']), cmap='twilight', levels=45, vmin=-40, vmax=40)
axs[1,1].set_title(r'$\tau \equiv \mathcal{M}_\text{on}^\text{state} (\text{lg})$', fontsize=15)
axs[1,2].contourf(eq_coarse.X, eq_coarse.Y, from_s(metrics_state_ref['om_s']), cmap='twilight', levels=45, vmin=-40, vmax=40)
axs[1,2].set_title(r'$\tau \equiv \mathcal{M}_{\text{on}}^\text{state} (\text{ref})$', fontsize=15)

axs[2,0].contourf(eq_coarse.X, eq_coarse.Y, from_s(metrics_subgrid_small['om_s']), cmap='twilight', levels=45, vmin=-40, vmax=40)
axs[2,0].set_title(r'$\tau \equiv \mathcal{M}_\text{on}^\text{subgrid} (\text{sm})$', fontsize=15)
axs[2,1].contourf(eq_coarse.X, eq_coarse.Y, from_s(metrics_subgrid_large['om_s']), cmap='twilight', levels=45, vmin=-40, vmax=40)
axs[2,1].set_title(r'$\tau \equiv \mathcal{M}_\text{on}^\text{subgrid} (\text{lg})$', fontsize=15)
axs[2,2].contourf(eq_coarse.X, eq_coarse.Y, from_s(metrics_subgrid_ref['om_s']), cmap='twilight', levels=45, vmin=-40, vmax=40)
axs[2,2].set_title(r'$\tau \equiv \mathcal{M}_{\text{on}}^\text{subgrid} (\text{ref})$', fontsize=15)

[axs[0,i].set_xticks([]) for i in range(3)]
[axs[1,i].set_xticks([]) for i in range(3)]
[axs[0,i].set_yticks([]) for i in range(1, 3)]
[axs[1,i].set_yticks([]) for i in range(1, 3)]
[axs[2,i].set_yticks([]) for i in range(1, 3)]
axs[-1,0].set_xlabel(r'$x$', fontsize=15)
axs[-1,1].set_xlabel(r'$x$', fontsize=15)
axs[-1,2].set_xlabel(r'$x$', fontsize=15)
axs[0,0].set_ylabel(r'$y$', fontsize=15)
axs[1,0].set_ylabel(r'$y$', fontsize=15)
axs[2,0].set_ylabel(r'$y$', fontsize=15)

fig.colorbar(contour, ax=axs, fraction=0.02, aspect=50, pad=0.03)

fig.savefig(os.path.join(data_path, 'vorticity_end.png'), dpi=200, bbox_inches='tight')
plt.show()

### A posteriori temporally averaged spectral metrics $E(k)$ and $Z(k)$:

In [ ]:
fig, axs = plt.subplots(ncols=2, nrows=1, figsize=(7.5, 3.5), dpi=120)

kr = metrics_dns['kr']
bm = metrics_dns['bin_max']
bm_c = metrics_nop['bin_max']

axs[0].loglog(kr[:bm], np.mean(metrics_dns['ek_k_t'], axis=0)[:bm], color='k', label=r'$\text{DNS}$')
axs[0].loglog(kr_c[:bm_c], np.mean(metrics_nop['ek_k_t'], axis=0)[:bm_c], label=r'$\tau \equiv 0$', linestyle='--', color='#999999')
axs[0].loglog(kr_c[:bm_c], np.mean(metrics_subgrid_small['ek_k_t'], axis=0)[:bm_c], label=r'$\tau \equiv \mathcal{M}_\text{on}^\text{subgrid} (\text{sm})$', color='#E69F00')
axs[0].loglog(kr_c[:bm_c], np.mean(metrics_state_large['ek_k_t'], axis=0)[:bm_c], label=r'$\tau \equiv \mathcal{M}_\text{on}^\text{state} (\text{lg})$', color='#D55E00', linestyle='--')
axs[0].loglog(kr_c[:bm_c], np.mean(metrics_subgrid_large['ek_k_t'], axis=0)[:bm_c], label=r'$\tau \equiv \mathcal{M}_\text{on}^\text{subgrid} (\text{lg})$', color='#D55E00')
axs[0].loglog(kr_c[:bm_c], np.mean(metrics_state_ref['ek_k_t'], axis=0)[:bm_c], label=r'$\tau \equiv \mathcal{M}_{\text{on}}^\text{state} (\text{ref})$', color='#56B4E9', linestyle='--')
axs[0].loglog(kr_c[:bm_c], np.mean(metrics_subgrid_ref['ek_k_t'], axis=0)[:bm_c], label=r'$\tau \equiv \mathcal{M}_{\text{on}}^\text{subgrid} (\text{ref})$', color='#56B4E9')

axs[0].axvline(k_f, color='lightgray', linestyle='--', label=r'$k_f$')

axs[0].set_ylim(bottom=1e-12)
axs[0].set_xlabel(r'$k$', fontsize=15)
axs[0].set_ylabel(r'$E(k)$', fontsize=15)
axs[0].legend(fontsize=10, frameon=False)
axs[0].tick_params(reset=True, axis='both', which='both', direction='in')

axs[1].loglog(kr[:bm], np.mean(metrics_dns['en_k_t'], axis=0)[:bm], color='k', label=r'$\text{DNS}$')
axs[1].loglog(kr_c[:bm_c], np.mean(metrics_nop['en_k_t'], axis=0)[:bm_c], label=r'$\tau \equiv 0$', linestyle='--', color='#999999')
axs[1].loglog(kr_c[:bm_c], np.mean(metrics_subgrid_small['en_k_t'], axis=0)[:bm_c], label=r'$\tau \equiv \mathcal{M}_\text{on}^\text{subgrid} (\text{sm})$', color='#E69F00')
axs[1].loglog(kr_c[:bm_c], np.mean(metrics_state_large['en_k_t'], axis=0)[:bm_c], label=r'$\tau \equiv \mathcal{M}_\text{on}^\text{state} (\text{lg})$', color='#D55E00', linestyle='--')
axs[1].loglog(kr_c[:bm_c], np.mean(metrics_subgrid_large['en_k_t'], axis=0)[:bm_c], label=r'$\tau \equiv \mathcal{M}_\text{on}^\text{subgrid} (\text{lg})$', color='#D55E00')
axs[1].loglog(kr_c[:bm_c], np.mean(metrics_state_ref['en_k_t'], axis=0)[:bm_c], label=r'$\tau \equiv \mathcal{M}_{\text{on}}^\text{state} (\text{ref})$', color='#56B4E9', linestyle='--')
axs[1].loglog(kr_c[:bm_c], np.mean(metrics_subgrid_ref['en_k_t'], axis=0)[:bm_c], label=r'$\tau \equiv \mathcal{M}_{\text{on}}^\text{subgrid} (\text{ref})$', color='#56B4E9')

axs[1].axvline(k_f, color='lightgray', linestyle='--')

axs[1].set_xlabel(r'$k$', fontsize=15)
axs[1].set_ylabel(r'$Z(k)$', fontsize=15)
#axs[1].legend(fontsize=10, frameon=False)
axs[1].tick_params(reset=True, axis='both', which='both', direction='in')

fig.tight_layout()
fig.savefig(os.path.join(data_path, 'spectra.pdf'))
plt.show()

### Distributions of vorticity and kinetic energy at the end of the a posteriori evaluation:

In [ ]:
fig, axs = plt.subplots(ncols=2, nrows=1, figsize=(7.0, 3.5), dpi=120)

om_g = from_s(metrics_fdns['om_s'])
om_g_range = 0.75 * max(om_g.max(), abs(om_g.min())) # 75% (avoid outliers)
sample_space = np.linspace(-om_g_range, om_g_range, num=500)


axs[0].plot(sample_space, 
            scipy.stats.gaussian_kde(from_s(metrics_fdns['om_s']).flatten()).pdf(sample_space), 
            color='k', 
            label=r'$\text{DNS}$'
           )
axs[0].plot(sample_space, 
            scipy.stats.gaussian_kde(from_s(metrics_nop['om_s']).flatten()).pdf(sample_space),
            label=r'$\tau \equiv 0$',
            linestyle='--',
            color='#999999'
           )
axs[0].plot(sample_space, 
            scipy.stats.gaussian_kde(from_s(metrics_subgrid_small['om_s']).flatten()).pdf(sample_space),
            label=r'$\tau \equiv \mathcal{M}_\text{on}^\text{subgrid} (\text{sm})$',
            color='#E69F00'
           )
axs[0].plot(sample_space, 
            scipy.stats.gaussian_kde(from_s(metrics_state_large['om_s']).flatten()).pdf(sample_space),
            label=r'$\tau \equiv \mathcal{M}_\text{on}^\text{state} (\text{lg})$',
            color='#D55E00',
            linestyle='--'
           )
axs[0].plot(sample_space, 
            scipy.stats.gaussian_kde(from_s(metrics_subgrid_large['om_s']).flatten()).pdf(sample_space),
            label=r'$\tau \equiv \mathcal{M}_\text{on}^\text{subgrid} (\text{lg})$',
            color='#D55E00'
           )
axs[0].plot(sample_space, 
            scipy.stats.gaussian_kde(from_s(metrics_state_ref['om_s']).flatten()).pdf(sample_space),
            label=r'$\tau \equiv \mathcal{M}_{\text{on}}^\text{state} (\text{ref})$',
            color='#56B4E9',
            linestyle='--'
           )
axs[0].plot(sample_space, 
            scipy.stats.gaussian_kde(from_s(metrics_subgrid_ref['om_s']).flatten()).pdf(sample_space),
            label=r'$\tau \equiv \mathcal{M}_\text{on}^\text{subgrid} (\text{ref})$',
            color='#56B4E9'
           )

x1, x2, y1, y2 = 19, 24, 0.001, 0.005
axins = axs[0].inset_axes([0.05, 0.4, 0.35, 0.35], xlim=(x1, x2), ylim=(y1, y2))
axins.tick_params(axis='both', which='both', labelleft=False, labelbottom=False)
axins.plot(sample_space, 
            scipy.stats.gaussian_kde(from_s(metrics_fdns['om_s']).flatten()).pdf(sample_space), 
            color='k', 
            label=r'$\text{DNS}$'
           )
axins.plot(sample_space, 
            scipy.stats.gaussian_kde(from_s(metrics_subgrid_large['om_s']).flatten()).pdf(sample_space),
            label=r'$\tau \equiv \mathcal{M}_\text{on}^\text{subgrid} (\text{lg})$',
            color='#D55E00'
           )
axins.plot(sample_space, 
            scipy.stats.gaussian_kde(from_s(metrics_state_ref['om_s']).flatten()).pdf(sample_space),
            label=r'$\tau \equiv \mathcal{M}_\text{on}^\text{state} (\text{ref})$',
            color='#56B4E9',
            linestyle='--'
           )
axins.plot(sample_space, 
            scipy.stats.gaussian_kde(from_s(metrics_subgrid_ref['om_s']).flatten()).pdf(sample_space),
            label=r'$\tau \equiv \mathcal{M}_\text{on}^\text{subgrid} (\text{ref})$',
            color='#56B4E9'
           )
axs[0].indicate_inset_zoom(axins, edgecolor='black')

eke_g = 0.75 * (from_s(metrics_fdns['ux_s']) + from_s(metrics_fdns['uy_s']))**2
sample_space = np.linspace(eke_g.min(), 0.2 * eke_g.max(), num=500)

def eke_pdf(ux_s, uy_s):
    ux_g = from_s(ux_s)
    uy_g = from_s(uy_s)
    eke_g = 0.5 * (ux_g + uy_g)**2
    return scipy.stats.gaussian_kde(eke_g.flatten()).pdf(
        sample_space
    )

axs[1].plot(sample_space, 
            eke_pdf(metrics_fdns['ux_s'], metrics_fdns['uy_s']), 
            color='k', 
            label=r'$\text{DNS}$'
           )
axs[1].plot(sample_space, 
            eke_pdf(metrics_nop['ux_s'], metrics_nop['uy_s']),
            label=r'$\tau \equiv 0$',
            linestyle='--',
            color='#999999'
           )
axs[1].plot(sample_space, 
            eke_pdf(metrics_subgrid_small['ux_s'], metrics_subgrid_small['uy_s']),
            label=r'$\tau \equiv \mathcal{M}_\text{on}^\text{subgrid} (\text{sm})$',
            color='#E69F00'
           )
axs[1].plot(sample_space, 
            eke_pdf(metrics_state_large['ux_s'], metrics_state_large['uy_s']),
            label=r'$\tau \equiv \mathcal{M}_\text{on}^\text{state} (\text{lg})$',
            color='#D55E00',
            linestyle='--'
           )
axs[1].plot(sample_space, 
            eke_pdf(metrics_subgrid_large['ux_s'], metrics_subgrid_large['uy_s']),
            label=r'$\tau \equiv \mathcal{M}_\text{on}^\text{subgrid} (\text{lg})$',
            color='#D55E00'
           )
axs[1].plot(sample_space, 
            eke_pdf(metrics_state_ref['ux_s'], metrics_state_ref['uy_s']),
            label=r'$\tau \equiv \mathcal{M}_{\text{on}}^\text{state} (\text{ref})$',
            color='#56B4E9',
            linestyle='--'
           )
axs[1].plot(sample_space, 
            eke_pdf(metrics_subgrid_ref['ux_s'], metrics_subgrid_ref['uy_s']),
            label=r'$\tau \equiv \mathcal{M}_\text{on}^\text{subgrid} (\text{ref})$',
            color='#56B4E9'
           )

axs[0].set_xlabel(r'$\omega(x, y)$', fontsize=15)
axs[1].set_xlabel(r'$E(x, y)$', fontsize=15)
axs[0].set_ylabel(r'$pdf$', fontsize=15)

#axs[0].legend(fontsize=11, frameon=False)
axs[1].legend(fontsize=10, frameon=False)
axs[0].tick_params(reset=True, axis='both', which='both', direction='in')
axs[1].tick_params(reset=True, axis='both', which='both', direction='in')

fig.tight_layout()
fig.savefig(os.path.join(data_path, 'distributions.pdf'))
plt.show()

### Similarity metrics:

In [ ]:
import pprint
print('=== Similarity scores ===')
print('state (ref)')
pprint.pprint(similarity_state_ref)
print('subgrid (sm)')
pprint.pprint(similarity_subgrid_small)
print('state (lg)')
pprint.pprint(similarity_state_large)
print('subgrid (lg)')
pprint.pprint(similarity_subgrid_large)
print('subgrid (ref)')
pprint.pprint(similarity_subgrid_ref)